In [ ]:
{ "nbformat": 4, "nbformat_minor": 5, "metadata": { "kernelspec": { "name": "python3", "display_name": "Python 3" }, "language_info": { "name": "python" } }, "cells": [ { "cell_type": "markdown", "metadata": {}, "source": [ "# Transcription, Diarization and Hindi↔English Translation (Kaggle / Colab)\n", "\n", "Two-stage pipeline focused on high-quality transcription and diarization for completed audio/video files. Stage 1: transcription + speaker labeling (diarization). Stage 2 (optional): Hindi ↔ English translation.\n", "\n", "Notes:\n", "- Kaggle kernels can't access a user's microphone/webcam from the cloud runtime; record locally and upload to the notebook or use Colab with the browser-based recording options.\n", "- Optional heavy dependencies (NeMo for diarization, IndicTrans2 translation) are disabled by default — enable only if you have GPU, enough disk, and time to download models.\n", "- The notebook writes outputs under /kaggle/working/outputs by default.\n" ] }, { "cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": [ "# Cell 1 — Install dependencies (run once). Comment/uncomment optional installs as needed.\n", "\n", "# Minimal ASR + audio\n", "!pip -q install --upgrade openai-whisper soundfile==0.13.1 numpy==1.26.4 ffmpeg-python==0.2.0\n", "\n", "# Optional translation dependencies (IndicTrans2)\n", "!pip -q install --upgrade transformers==4.57.3 sentencepiece==0.2.1 IndicTransToolkit==1.1.1\n", "\n", "# Optional: NeMo for diarization (heavy; enable only on GPU/Colab with enough disk)\n", "# !pip -q install --upgrade "nemo_toolkit[asr]==2.7.3"\n", "\n", "print('Cell 1 complete: packages installed (or already present).')" ] }, { "cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": [ "# Cell 2 — Imports, configuration, and global paths\n", "import os, json, time, uuid, math, re, gc, subprocess, logging\n", "from pathlib import Path\n", "from zipfile import ZipFile, ZIP_DEFLATED\n", "import numpy as np\n", "import soundfile as sf\n", "import torch\n", "\n", "# Paths\n", "MODEL_ROOT = Path('/kaggle/working/models')\n", "OUTPUT_ROOT = Path('/kaggle/working/outputs')\n", "MODEL_ROOT.mkdir(parents=True, exist_ok=True)\n", "OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)\n", "\n", "# Audio constants\n", "SAMPLE_RATE = 16000\n", "MIN_MEDIA_SECONDS = 0.5\n", "MAX_MEDIA_SECONDS = 60 * 60\n", "\n", "# Models (default IDs — configurable)\n", "ASR_MODEL_ID = 'large-v3' # Whisper\n", "EN_HI_MODEL_ID = 'naklitechie/indictrans2-en-indic-dist-200M'\n", "HI_EN_MODEL_ID = 'prajdabre/rotary-indictrans2-indic-en-dist-200M'\n", "\n", "# Flags\n", "CONTEXT_CORRECTIONS_ENABLED = False\n", "ENABLE_NEMO_DIARIZATION = False\n", "DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'\n", "\n", "# Logging\n", "logging.basicConfig(level=logging.INFO)\n", "logger = logging.getLogger('pipeline')\n", "print(f'Cell 2: DEVICE={DEVICE}, MODEL_ROOT={MODEL_ROOT}, OUTPUT_ROOT={OUTPUT_ROOT}')" ] }, { "cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": [ "# Cell 3 — Utilities (normalization, timestamps, zip helper)\n", "import unicodedata\n", "def normalize_spacing(text):\n", " text = unicodedata.normalize('NFKC', str(text or ''))\n", " text = re.sub(r'\s+', ' ', text).strip()\n", " return re.sub(r"\s+([,.;:!?।])", r"\1", text)\n", "\n", "def format_clock(seconds):\n", " milliseconds = int(round(max(0.0, float(seconds)) * 1000))\n", " hours, remainder = divmod(milliseconds, 3_600_000)\n", " minutes, remainder = divmod(remainder, 60_000)\n", " secs, millis = divmod(remainder, 1000)\n", " return f"{hours:02d}:{minutes:02d}:{secs:02d}.{millis:03d}"\n", "\n", "def srt_timestamp(seconds):\n", " milliseconds = int(round(max(0.0, float(seconds)) * 1000))\n", " hours, remainder = divmod(milliseconds, 3_600_000)\n", " minutes, remainder = divmod(remainder, 60_000)\n", " secs, millis = divmod(remainder, 1000)\n", " return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"\n", "\n", "def zip_job_dir(job_dir: Path):\n", " zip_path = job_dir / 'results.zip'\n", " with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as z:\n", " for p in sorted(job_dir.iterdir()):\n", " if p.is_file() and p.name != zip_path.name:\n", " z.write(p, arcname=p.name)\n", " return zip_path\n", "\n", "print('Cell 3: utilities loaded')" ] }, { "cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": [ "# Cell 4 — Audio I/O: resolve path, probe media (ffprobe), convert to 16 kHz mono WAV and waveform metrics\n", "def resolve_media_path(media_value):\n", " # Accept a path string or Path or an upload dict\n", " if isinstance(media_value, dict):\n", " candidate = media_value.get('path') or media_value.get('name') or media_value.get('file')\n", " else:\n", " candidate = media_value\n", " if not candidate:\n", " raise FileNotFoundError('No media provided')\n", " path = Path(str(candidate))\n", " if not path.exists():\n", " raise FileNotFoundError(f'File not found: {path}')\n", " print(f'Found media file: {path}')\n", " return path\n", "\n", "def probe_media(path: Path):\n", " cmd = [\n", " 'ffprobe', '-v', 'error', '-select_streams', 'a:0',\n", " '-show_entries', 'stream=codec_name,channels,channel_layout,sample_rate,duration',\n", " '-show_entries', 'format=duration', '-of', 'json', str(path),\n", " ]\n", " result = subprocess.run(cmd, capture_output=True, text=True)\n", " if result.returncode != 0:\n", " raise RuntimeError('ffprobe failed: ' + (result.stderr or result.stdout))\n", " payload = json.loads(result.stdout or '{}')\n", " streams = payload.get('streams') or []\n", " if not streams:\n", " raise RuntimeError('No audio stream found')\n", " stream = streams[0]\n", " duration = float(stream.get('duration') or payload.get('format', {}).get('duration') or 0.0)\n", " meta = {\n", " 'codec': stream.get('codec_name'),\n", " 'channels': int(stream.get('channels') or 1),\n", " 'channel_layout': stream.get('channel_layout') or 'unknown',\n", " 'source_sample_rate': int(stream.get('sample_rate') or 0),\n", " 'duration_seconds': duration,\n", " }\n", " print(f'Probe result: duration={meta["duration_seconds"]:.2f}s, channels={meta["channels"]}, sample_rate={meta["source_sample_rate"]}')\n", " return meta\n", "\n", "def run_ffmpeg_to_16k_mono(source: Path, out_path: Path, enhanced=False):\n", " filters = [f'aresample={SAMPLE_RATE}']\n", " if enhanced:\n", " filters.extend(['highpass=f=65', 'lowpass=f=7600', 'dynaudnorm=f=250'])\n", " af = ','.join(filters)\n", " cmd = [\n", " 'ffmpeg', '-y', '-hide_banner', '-loglevel', 'error', '-i', str(source), '-vn',\n", " '-af', af, '-ac', '1', '-ar', str(SAMPLE_RATE), '-c:a', 'pcm_s16le', str(out_path)\n", " ]\n", " print(f'Running ffmpeg -> {out_path} (enhanced={enhanced})')\n", " result = subprocess.run(cmd, capture_output=True, text=True)\n", " if result.returncode != 0 or not out_path.exists():\n", " raise RuntimeError('ffmpeg error: ' + (result.stderr or result.stdout))\n", " print(f'ffmpeg completed: {out_path} created')\n", "\n", "def waveform_metrics(path: Path):\n", " audio, sr = sf.read(str(path), dtype='float32', always_2d=False)\n", " audio = np.asarray(audio, dtype=np.float32).reshape(-1)\n", " if sr != SAMPLE_RATE:\n", " raise RuntimeError('Unexpected sample rate in prepared file')\n", " rms = float(np.sqrt(np.mean(np.square(audio)) + 1e-12))\n", " peak = float(np.max(np.abs(audio)))\n", " frame_size = max(1, int(0.10 * sr))\n", " frame_count = len(audio) // frame_size\n", " if frame_count:\n", " frames = audio[:frame_count * frame_size].reshape(frame_count, frame_size)\n", " frame_rms = np.sqrt(np.mean(np.square(frames), axis=1) + 1e-12)\n", " frame_db = 20.0 * np.log10(np.maximum(frame_rms, 1e-8))\n", " noise_floor = float(np.percentile(frame_db, 20))\n", " speech_level = float(np.percentile(frame_db, 90))\n", " else:\n", " noise_floor = speech_level = 20.0 * math.log10(max(rms, 1e-8))\n", " metrics = {\n", " 'rms_dbfs': round(20.0 * math.log10(max(rms, 1e-8)), 2),\n", " 'peak': round(peak, 4),\n", " 'clipping_fraction': float(np.mean(np.abs(audio) >= 0.999)),\n", " 'approx_noise_floor_dbfs': round(noise_floor, 2),\n", " 'approx_speech_level_dbfs': round(speech_level, 2),\n", " 'approx_snr_db': round(max(0.0, speech_level - noise_floor), 2),\n", " }\n", " print(f'Waveform metrics: SNR={metrics["approx_snr_db"]} dB, RMS={metrics["rms_dbfs"]} dBFS')\n", " return metrics, audio\n", "\n", "print('Cell 4 ready: audio IO helpers loaded')" ] }, { "cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": [ "# Cell 5 — prepare_media: create job dir, convert audio to 16k mono and produce tracks\n", "def prepare_media(media_value):\n", " source = resolve_media_path(media_value)\n", " meta = probe_media(source)\n", " job_id = time.strftime('job-%Y%m%d-%H%M%S-') + uuid.uuid4().hex[:6]\n", " job_dir = OUTPUT_ROOT / job_id\n", " audio_dir = job_dir / 'audio'\n", " audio_dir.mkdir(parents=True, exist_ok=True)\n", " mono_raw = audio_dir / 'mono_raw.wav'\n", " run_ffmpeg_to_16k_mono(source, mono_raw, enhanced=False)\n", " metrics, audio = waveform_metrics(mono_raw)\n", " if meta['duration_seconds'] == 0.0:\n", " meta['duration_seconds'] = len(audio) / SAMPLE_RATE\n", " if meta['duration_seconds'] < MIN_MEDIA_SECONDS:\n", " raise RuntimeError('Media too short')\n", " mono_enhanced = audio_dir / 'mono_enhanced.wav'\n", " tracks = [{\n", " 'speaker': None,\n", " 'raw_path': mono_raw,\n", " 'enhanced_path': mono_enhanced,\n", " 'raw_metrics': metrics,\n", " 'diarization_path': mono_raw,\n", " 'diarization_metrics': metrics,\n", " }]\n", " meta['channel_strategy'] = 'mono_diarization'\n", " meta['source_file'] = source.name\n", " print(f'prepare_media: job_dir={job_dir}, duration={meta["duration_seconds"]:.2f}s')\n", " return source, job_dir, tracks, meta\n" ] }, { "cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": [ "# Cell 6 — Load models (Whisper + optional translation + optional NeMo)\n", "import whisper\n", "from transformers import AutoModelForSeq2SeqLM, AutoTokenizer\n", "\n", "ASR_MODEL = None\n", "EN_HI_MODEL = None\n", "EN_HI_TOKENIZER = None\n", "HI_EN_MODEL = None\n", "HI_EN_TOKENIZER = None\n", "DIAR_MODEL = None\n", "\n", "def load_models(load_translation=True, load_diarization=False):\n", " global ASR_MODEL, EN_HI_MODEL, EN_HI_TOKENIZER, HI_EN_MODEL, HI_EN_TOKENIZER, DIAR_MODEL\n", " if ASR_MODEL is None:\n", " print('Loading Whisper ASR model (this may take a few minutes depending on network)')\n", " ASR_MODEL = whisper.load_model(ASR_MODEL_ID, device=DEVICE, download_root=str(MODEL_ROOT/'whisper')).eval()\n", " print('Whisper loaded')\n", " if load_translation:\n", " if EN_HI_MODEL is None:\n", " print('Loading IndicTrans2 EN->HI models (this may be large)')\n", " EN_HI_TOKENIZER = AutoTokenizer.from_pretrained(EN_HI_MODEL_ID, trust_remote_code=True, cache_dir=str(MODEL_ROOT/'en_hi'))\n", " EN_HI_MODEL = AutoModelForSeq2SeqLM.from_pretrained(EN_HI_MODEL_ID, trust_remote_code=True, torch_dtype=torch.float16, cache_dir=str(MODEL_ROOT/'en_hi')).to(DEVICE).eval()\n", " print('EN->HI model loaded')\n", " if HI_EN_MODEL is None:\n", " print('Loading IndicTrans2 HI->EN models (this may be large)')\n", " HI_EN_TOKENIZER = AutoTokenizer.from_pretrained(HI_EN_MODEL_ID, trust_remote_code=True, cache_dir=str(MODEL_ROOT/'hi_en'))\n", " HI_EN_MODEL = AutoModelForSeq2SeqLM.from_pretrained(HI_EN_MODEL_ID, trust_remote_code=True, torch_dtype=torch.float16, cache_dir=str(MODEL_ROOT/'hi_en')).to(DEVICE).eval()\n", " print('HI->EN model loaded')\n", " if load_diarization:\n", " if not ENABLE_NEMO_DIARIZATION:\n", " print('NeMo diarization is disabled by ENABLE_NEMO_DIARIZATION flag')\n", " return\n", " try:\n", " from nemo.collections.asr.models import SortformerEncLabelModel\n", " if DIAR_MODEL is None:\n", " print('Loading NeMo diarization model (heavy, requires GPU & deps)')\n", " DIAR_MODEL = SortformerEncLabelModel.from_pretrained('nvidia/diar_streaming_sortformer_4spk-v2.1').to(DEVICE).eval()\n", " print('NeMo diarization loaded')\n", " except Exception as e:\n", " print('Failed to load NeMo diarization:', e)\n", " raise\n", "\n", "print('Cell 6 ready: model loader defined (models not loaded until called)')" ] }, { "cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": [ "# Cell 7 — ASR functions: transcribe_candidate and a simplified adaptive selection\n", "def transcribe_candidate(wav_path: Path, requested_language=None, beam_size=2, initial_prompt=None):\n", " beam_size = max(2, int(beam_size))\n", " print(f'Transcribing {wav_path} (requested_language={requested_language}, beam={beam_size})')\n", " result = ASR_MODEL.transcribe(\n", " str(wav_path),\n", " language=requested_language,\n", " task='transcribe',\n", " beam_size=beam_size,\n", " temperature=0.0,\n", " condition_on_previous_text=False,\n", " word_timestamps=True,\n", " fp16=(DEVICE=='cuda'),\n", " )\n", " print('Whisper pass complete — detected language:', result.get('language'), 'segments:', len(result.get('segments', [])))\n", " segments = []\n", " for seg in result.get('segments', []):\n", " text = normalize_spacing(seg.get('text', ''))\n", " if not text:\n", " continue\n", " words = []\n", " for w in seg.get('words', []) or []:\n", " wt = normalize_spacing(w.get('word', ''))\n", " if not wt:\n", " continue\n", " words.append({\n", " 'start_seconds': round(float(w.get('start', seg.get('start', 0.0))), 3),\n", " 'end_seconds': round(float(w.get('end', seg.get('end', seg.get('start', 0.0)))), 3),\n", " 'text': wt,\n", " 'probability': float(w.get('probability', 0.0)),\n", " })\n", " segments.append({\n", " 'start_seconds': round(float(seg['start']), 3),\n", " 'end_seconds': round(float(seg['end']), 3),\n", " 'text': text,\n", " 'avg_logprob': float(seg.get('avg_logprob', 0.0)),\n", " 'no_speech_prob': float(seg.get('no_speech_prob', 0.0)),\n", " 'compression_ratio': float(seg.get('compression_ratio', 0.0)),\n", " 'words': words,\n", " })\n", " candidate = {\n", " 'variant': 'raw',\n", " 'requested_language': requested_language or 'auto',\n", " 'detected_language': result.get('language'),\n", " 'segments': segments,\n", " 'text': normalize_spacing(' '.join(s['text'] for s in segments)),\n", " 'beam_size': beam_size,\n", " }\n", " candidate['quality_score'] = float(np.mean([seg.get('avg_logprob', -10.0) for seg in segments])) if segments else -999.0\n", " print(f'Candidate quality_score={candidate["quality_score"]:.3f}, text_length={len(candidate["text"]) }')\n", " return candidate\n", "\n", "def candidate_needs_retry(candidate, track):\n", " if candidate.get('quality_score', -999.0) < -1.5:\n", " print('Candidate low quality -> will retry with enhanced audio')\n", " return True\n", " if float(track['raw_metrics'].get('approx_snr_db', 99.0)) < 12.0:\n", " print('Track SNR < 12dB -> will retry with enhanced audio')\n", " return True\n", " return False\n", "\n", "def transcribe_whole_file(track, profile='Balanced'):\n", " candidates = []\n", " raw_path = Path(track['raw_path'])\n", " print('Starting primary ASR pass (raw)')\n", " candidates.append(transcribe_candidate(raw_path, requested_language=None, beam_size=2))\n", " if candidate_needs_retry(candidates[0], track):\n", " enhanced = Path(track['enhanced_path'])\n", " if not enhanced.exists():\n", " run_ffmpeg_to_16k_mono(raw_path, enhanced, enhanced=True)\n", " print('Starting enhanced ASR pass (enhanced audio)')\n", " candidates.append(transcribe_candidate(enhanced, requested_language=None, beam_size=2))\n", " usable = [c for c in candidates if c['segments']]\n", " if not usable:\n", " raise RuntimeError('No speech detected by Whisper in any pass')\n", " selected = max(usable, key=lambda x: x.get('quality_score', -999.0))\n", " print('Selected candidate variant=', selected.get('variant'), 'quality=', selected.get('quality_score'))\n", " return selected, candidates\n", "\n", "print('Cell 7 ready: ASR functions defined')" ] }, { "cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": [ "# Cell 8 — Diarization and a robust fallback\n", "def run_nemo_diarization(wav_path: Path):\n", " if DIAR_MODEL is None:\n", " raise RuntimeError('NeMo diarization model not loaded')\n", " print(f'Running NeMo diarization on: {wav_path}')\n", " audio, sr = sf.read(str(wav_path), dtype='float32', always_2d=False)\n", " audio = np.asarray(audio, dtype=np.float32).reshape(-1)\n", " if sr != SAMPLE_RATE:\n", " raise RuntimeError('NeMo input must be 16 kHz')\n", " with torch.inference_mode():\n", " predicted = DIAR_MODEL.diarize(audio=[np.clip(audio, -1.0, 1.0)], sample_rate=SAMPLE_RATE, batch_size=1, postprocessing_yaml=None, num_workers=0, verbose=False)\n", " lines = predicted[0] if isinstance(predicted, (list, tuple)) else predicted\n", " timeline = []\n", " for item in lines:\n", " if isinstance(item, (list, tuple)) and len(item) >= 3:\n", " start, end, label = float(item[0]), float(item[1]), str(item[2])\n", " else:\n", " parts = str(item).strip().split()\n", " if len(parts) < 3:\n", " continue\n", " start, end, label = float(parts[0]), float(parts[1]), parts[2]\n", " timeline.append({'start_seconds': round(start,3), 'end_seconds': round(end,3), 'speaker': label})\n", " print(f'NeMo diarization produced {len(timeline)} segments')\n", " return timeline\n", "\n", "def simple_speaker_assignment(candidate):\n", " # fallback single-speaker assignment with naive gap-based turn merging\n", " print('Running simple speaker assignment fallback (all speech -> Speaker 1)')\n", " words = []\n", " for s in candidate['segments']:\n", " if s['words']:\n", " for w in s['words']:\n", " words.append(dict(w))\n", " else:\n", " words.append({'start_seconds': s['start_seconds'], 'end_seconds': s['end_seconds'], 'text': s['text'], 'probability': math.exp(s.get('avg_logprob', -10.0))})\n", " records = []\n", " for w in words:\n", " records.append({\n", " 'start_seconds': w['start_seconds'],\n", " 'end_seconds': w['end_seconds'],\n", " 'speaker': 'Speaker 1',\n", " 'transcript': w['text'],\n", " 'word_confidence': round(float(w.get('probability', 0.0)), 4),\n", " 'active_speakers': ['Speaker 1'],\n", " 'overlap_detected': False,\n", " 'speaker_assignment_uncertain': False,\n", " 'source_track_index': 0,\n", " 'whisper_language': candidate.get('detected_language'),\n", " 'whisper_requested_language': candidate.get('requested_language'),\n", " })\n", " # merge consecutive words into turns\n", " turns = []\n", " gap_threshold = 0.75\n", " for item in records:\n", " if not turns:\n", " turns.append(dict(item))\n", " continue\n", " prev = turns[-1]\n", " if item['start_seconds'] - prev['end_seconds'] <= gap_threshold:\n", " prev['end_seconds'] = item['end_seconds']\n", " prev['transcript'] = normalize_spacing(prev['transcript'] + ' ' + item['transcript'])\n", " prev['word_confidence'] = round(float(np.mean([prev['word_confidence'], item['word_confidence']])), 4)\n", " else:\n", " turns.append(dict(item))\n", " print(f'simple_speaker_assignment -> {len(turns)} turns')\n", " return turns\n", "\n", "print('Cell 8 ready: diarization helpers defined')" ] }, { "cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": [ "# Cell 9 — Postprocessing: clamp timestamps, merge short gaps, format conversation, export\n", "def merge_intervals(intervals, maximum_gap=0.15):\n", " merged = []\n", " for s,e in sorted(intervals):\n", " s, e = float(s), float(e)\n", " if merged and s <= merged[-1][1] + maximum_gap:\n", " merged[-1][1] = max(merged[-1][1], e)\n", " else:\n", " merged.append([s,e])\n", " return merged\n", "\n", "def postprocess_records(records, media_duration):\n", " print('Postprocessing: clamp timestamps and remove too-short segments')\n", " cleaned = []\n", " stats = {'removed':0, 'merged':0}\n", " for r in sorted(records, key=lambda x:(x['start_seconds'], x['end_seconds'])):\n", " start = round(max(0.0, min(media_duration, float(r['start_seconds']))), 3)\n", " end = round(max(0.0, min(media_duration, float(r['end_seconds']))), 3)\n", " if end - start < 0.04:\n", " stats['removed'] += 1\n", " continue\n", " r['start_seconds'] = start\n", " r['end_seconds'] = end\n", " r['transcript'] = normalize_spacing(r.get('transcript',''))\n", " cleaned.append(r)\n", " merged = []\n", " for r in cleaned:\n", " if merged and merged[-1]['speaker'] == r['speaker'] and r['start_seconds'] - merged[-1]['end_seconds'] <= 1.2:\n", " merged[-1]['end_seconds'] = max(merged[-1]['end_seconds'], r['end_seconds'])\n", " merged[-1]['transcript'] = normalize_spacing(merged[-1]['transcript'] + ' ' + r['transcript'])\n", " stats['merged'] += 1\n", " continue\n", " merged.append(r)\n", " print(f'Postprocessing done: removed={stats["removed"]}, merged={stats["merged"]}, final_turns={len(merged)}')\n", " return merged, stats\n", "\n", "def format_conversation(records):\n", " groups = []\n", " for item in records:\n", " label = item['speaker'] + (' [uncertain]' if item.get('speaker_assignment_uncertain') else '')\n", " start = item['start_seconds']\n", " end = item['end_seconds']\n", " transcript = item.get('transcript','')\n", " if not groups:\n", " groups.append({'label': label, 'start_seconds': start, 'end_seconds': end, 'transcript': transcript})\n", " continue\n", " last = groups[-1]\n", " if last['label'] == label and start - last['end_seconds'] <= 3.5:\n", " last['end_seconds'] = max(last['end_seconds'], end)\n", " last['transcript'] = normalize_spacing(last['transcript'] + ' ' + transcript)\n", " else:\n", " groups.append({'label': label, 'start_seconds': start, 'end_seconds': end, 'transcript': transcript})\n", " transcript_text = '\n\n'.join(f"{g['label']}: {g['transcript']}" for g in groups)\n", " return transcript_text\n", "\n", "print('Cell 9 ready: postprocessing & formatting utilities loaded')" ] }, { "cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": [ "# Cell 10 — Translation wrappers (IndicTrans2). Load only if models available\n", "def translate_batch(texts, direction='en-hi', batch_size=8):\n", " if direction == 'en-hi':\n", " tokenizer = EN_HI_TOKENIZER\n", " model = EN_HI_MODEL\n", " else:\n", " tokenizer = HI_EN_TOKENIZER\n", " model = HI_EN_MODEL\n", " results = []\n", " for i in range(0, len(texts), batch_size):\n", " batch = texts[i:i+batch_size]\n", " encoded = tokenizer(batch, truncation=True, padding=True, return_tensors='pt').to(DEVICE)\n", " with torch.inference_mode():\n", " gen = model.generate(**encoded, max_new_tokens=384, num_beams=5, early_stopping=True)\n", " decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)\n", " results.extend([normalize_spacing(x) for x in decoded])\n", " print(f'Translated {len(texts)} turns direction={direction}')\n", " return results\n", "\n", "print('Cell 10: translation wrappers ready (models must be loaded first)')" ] }, { "cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": [ "# Cell 11 — High-level pipeline: process_file (Stage 1) and translate_stage2 (Stage 2)\n", "def process_file(media_value, processing_profile='Balanced', load_translation=False, load_diarization=False):\n", " print('Stage 1: prepare_media')\n", " source, job_dir, tracks, metadata = prepare_media(media_value)\n", " print('Stage 1: load models (ASR plus optional)')\n", " load_models(load_translation=load_translation, load_diarization=load_diarization)\n", " selected_candidates = []\n", " print('Stage 1: running ASR for each track')\n", " for track in tracks:\n", " selected, all_candidates = transcribe_whole_file(track, profile=processing_profile)\n", " selected_candidates.append(selected)\n", " # Diarization\n", " records = []\n", " if ENABLE_NEMO_DIARIZATION and DIAR_MODEL is not None:\n", " try:\n", " print('Stage 1: running NeMo diarization')\n", " timeline = run_nemo_diarization(tracks[0]['diarization_path'])\n", " words = []\n", " for seg in selected_candidates[0]['segments']:\n", " if seg['words']:\n", " for w in seg['words']:\n", " words.append(w)\n", " else:\n", " words.append({'start_seconds': seg['start_seconds'], 'end_seconds': seg['end_seconds'], 'text': seg['text'], 'probability': math.exp(seg.get('avg_logprob', -10.0))})\n", " for w in words:\n", " primary = 'Speaker uncertain'\n", " for t in timeline:\n", " if w['start_seconds'] >= t['start_seconds'] and w['end_seconds'] <= t['end_seconds']:\n", " primary = t['speaker']\n", " break\n", " records.append({\n", " 'start_seconds': w['start_seconds'],\n", " 'end_seconds': w['end_seconds'],\n", " 'speaker': primary,\n", " 'transcript': w['text'],\n", " 'word_confidence': round(float(w.get('probability',0.0)),4),\n", " 'active_speakers': [primary] if primary!='Speaker uncertain' else ['Speaker uncertain'],\n", " 'source_track_index': 0,\n", " })\n", " except Exception as e:\n", " print('NeMo diarization failed — falling back to simple assignment:', e)\n", " records = simple_speaker_assignment(selected_candidates[0])\n", " else:\n", " print('Using fallback simple speaker assignment')\n", " records = simple_speaker_assignment(selected_candidates[0])\n", " # Postprocess\n", " records, post_stats = postprocess_records(records, metadata['duration_seconds'])\n", " # Export results\n", " (job_dir / 'transcript.txt').write_text(format_conversation(records) + '\n', encoding='utf-8')\n", " (job_dir / 'records.json').write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')\n", " zip_path = zip_job_dir(job_dir)\n", " print(f'Stage 1 complete: job_dir={job_dir} zip={zip_path}')\n", " stage_state = {'job_dir': str(job_dir), 'records': records, 'tracks': tracks, 'metadata': metadata, 'asr_candidates': selected_candidates}\n", " return format_conversation(records), records, str(zip_path), stage_state\n", "\n", "def translate_stage2(stage_state, enable_audio_fallback=False):\n", " print('Stage 2: translate finalized turns')\n", " if not stage_state or 'records' not in stage_state:\n", " raise RuntimeError('Stage 1 results missing')\n", " load_models(load_translation=True, load_diarization=False)\n", " records = [dict(r) for r in stage_state['records']]\n", " texts_en_hi, texts_hi_en = [], []\n", " indices_en_hi, indices_hi_en = [], []\n", " for i, r in enumerate(records):\n", " txt = r.get('transcript','')\n", " devanagari = bool(re.search(r'[\u0900-\u097F]', txt))\n", " latin = bool(re.search(r'[A-Za-z]', txt))\n", " if devanagari and not latin:\n", " indices_hi_en.append(i); texts_hi_en.append(txt)\n", " elif latin and not devanagari:\n", " indices_en_hi.append(i); texts_en_hi.append(txt)\n", " else:\n", " whisper_lang = r.get('whisper_language')\n", " if whisper_lang == 'hi':\n", " indices_hi_en.append(i); texts_hi_en.append(txt)\n", " else:\n", " indices_en_hi.append(i); texts_en_hi.append(txt)\n", " translations = {}\n", " if texts_en_hi:\n", " out = translate_batch(texts_en_hi, direction='en-hi')\n", " for idx, t in zip(indices_en_hi, out):\n", " translations[idx] = t\n", " if texts_hi_en:\n", " out = translate_batch(texts_hi_en, direction='hi-en')\n", " for idx, t in zip(indices_hi_en, out):\n", " translations[idx] = t\n", " for i, r in enumerate(records):\n", " r['translation'] = translations.get(i, '')\n", " job_dir = Path(stage_state['job_dir'])\n", " (job_dir / 'translation.json').write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')\n", " zip_path = zip_job_dir(job_dir)\n", " print('Stage 2 complete — zip:', zip_path)\n", " return records, str(zip_path)\n", "\n", "print('Cell 11: pipeline Entrypoints ready')" ] }, { "cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": [ "# Cell 12 — Example run (edit example_path to your uploaded file path)\n", "example_path = '/kaggle/input/my-audio/example.wav' # change to your uploaded file path\n", "if Path(example_path).exists():\n", " print('Running pipeline on example:', example_path)\n", " transcript, records, zip_path, state = process_file(example_path, processing_profile='Balanced', load_translation=False, load_diarization=False)\n", " print('\n--- Transcript Preview ---\n')\n", " print(transcript[:2000])\n", " print('\nSaved zip:', zip_path)\n", "else:\n", " print('Example file not found — upload a file and set example_path accordingly')" ] }, { "cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": [ "# Cell 13 — small deterministic tests for utility functions\n", "def test_merge_intervals():\n", " inp = [(0.0, 1.0),(1.05,2.0),(3.0,3.2)]\n", " out = merge_intervals(inp, maximum_gap=0.1)\n", " assert out == [[0.0, 2.0],[3.0,3.2]]\n", "test_merge_intervals()\n", "print('merge_intervals test passed')\n", "\n", "def test_normalize_spacing():\n", " assert normalize_spacing('hello world !') == 'hello world!'\n", "test_normalize_spacing()\n", "print('normalize_spacing test passed')" ] }, { "cell_type": "markdown", "metadata": {}, "source": [ "## Validation summary — public references and best practices\n", "\n", "I checked public model cards and literature commonly used by practitioners. Key references and notes:\n", "- OpenAI Whisper large-v3 (Hugging Face model page): general-purpose multilingual ASR with strong performance across languages. See: https://huggingface.co/openai/whisper-large-v3\n", " - Reported WER varies by dataset; measure on your own held-out Hindi/English/Hinglish data to estimate production accuracy.\n", "- NVIDIA NeMo Sortformer diarization (model card): reported DER on CALLHOME-like datasets; Sortformer is designed for streaming diarization and overlapping speech handling. See: https://huggingface.co/nvidia/diar_streaming_sortformer_4spk-v2.1 and NVIDIA NeMo docs: https://docs.nvidia.com\n", " - NeMo's reported DER (on specific benchmarks/configs) is useful as a ceiling — real call audio (compression, channels, background) will change results.\n", "- IndicTrans2 translation models (model cards): test on your in-domain samples for quality; romanized Hindi often needs special handling or audio-based re-decode fallback.\n", "\n", "Best-practice validation steps I recommend for your data:\n", "1. Build a small representative test set (10–50 files) with human reference transcripts covering English, Hindi (Devanagari), and Hinglish (Romanized Hindi) including noisy/overlap examples.\n", "2. Run Stage 1 on those files and compute WER/CER using jiwer or similar; for diarization, compare speaker-time or RTTM against human annotations and compute DER.\n", "3. If Hindi→English translation is required, compute BLEU or direct human evaluation on translated outputs; watch for missing content in Romanized inputs.\n", "\n", "If you want, I can run a small automated check (on your provided sample files) and produce WER/CER/DER reports and per-file diagnostics.\n" ] } ] }